# Adaptive RAG Study Assistant — Step-by-Step Walkthrough

This notebook walks through the pipeline manually, one stage at a time,
so you can inspect intermediate outputs (chunks, embeddings, retrieval,
memory, chains) before it's wired together in `backend/pipeline.py`.

Run `python setup_index.py` once first if you haven't built the FAISS index yet.

In [ ]:
import sys
sys.path.append('..')

from backend.ingestion import load_and_chunk_slides
from backend.embeddings import get_or_build_vectorstore, get_retriever, build_vectorstore
from backend.prompts import get_prompt
from backend.memory import get_memory
from backend.chains import run_chain
from backend.pipeline import StudyAssistantPipeline, PipelineConfig

: 

## 1. Load & Chunk Slides

In [ ]:
chunks = load_and_chunk_slides()
print(f"Total chunks: {len(chunks)}")
print("\nSample chunk:\n", chunks[0].page_content[:400])

## 2. Embeddings + FAISS Index

In [ ]:
vectorstore = build_vectorstore(chunks)  # only needs to run once; reruns overwrite the index
retriever = get_retriever(vectorstore, k=4)

## 3. Semantic Search (Retrieval)

In [ ]:
test_question = "What is self-attention?"
results = retriever.invoke(test_question)
for r in results:
    print('---', r.metadata.get('source_file'), '---')
    print(r.page_content[:200], '\n')

## 4. Zero-Shot vs Few-Shot Prompts

In [ ]:
zero = get_prompt('zero-shot')
few = get_prompt('few-shot')
print(zero.format(history='(none)', context='[slide content here]', question=test_question))

## 5. Memory Types — compare Buffer vs Summary

Try asking a couple of follow-up questions with each memory type and
compare what `get_history_text()` returns.

In [ ]:
buffer_mem = get_memory('buffer')
buffer_mem.save_turn('What is RNN?', 'An RNN is a network with a hidden state...')
buffer_mem.save_turn('What about LSTM?', 'LSTM adds gates to control memory...')
print(buffer_mem.get_history_text())

## 6. Full Pipeline — try different configurations

This is the same class the Streamlit frontend uses. Change the
`PipelineConfig` values below and compare answers/pipelines.

In [ ]:
config = PipelineConfig(
    prompting_strategy='few-shot',   # 'zero-shot' | 'few-shot'
    memory_strategy='summary',        # 'buffer' | 'buffer-window' | 'summary' | 'entity'
    chain_type='refine',              # 'simple-sequential' | 'sequential' | 'mapreduce' | 'refine'
)
pipeline = StudyAssistantPipeline(config, session_id='notebook_demo')

result = pipeline.ask(test_question)
print('PIPELINE:', ' -> '.join(result.pipeline_steps))
print('\nANSWER:\n', result.answer)

## 7. MapReduce demo — summarizing a whole lecture

In [ ]:
config_mr = PipelineConfig(chain_type='mapreduce')
pipeline_mr = StudyAssistantPipeline(config_mr, session_id='notebook_mapreduce')
result_mr = pipeline_mr.ask('Summarize this lecture')
print(result_mr.answer)